In [1]:
pip install polars fsspec huggingface_hub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import polars as pl
import os

# ==========================================
# CẤU HÌNH DANH MỤC CHO NHÓM 3
# ==========================================

OUTPUT_DIR = r"D:\Kì II 2025-2026\BTL_BIGDATA\Data"

CATEGORIES = [
    "All_Beauty",
    "Amazon_Fashion",
    "Handmade_Products",
    "Health_and_Personal_Care",
    "Baby_Products",
    "Grocery_and_Gourmet_Food",
    "Sports_and_Outdoors",
    "Beauty_and_Personal_Care",
    "Health_and_Household",
    # "Clothing_Shoes_and_Jewelry" 
]

In [22]:
# ==========================================
# 2. HÀM XỬ LÝ CHÍNH
# ==========================================
def ingest_https_to_parquet(category):
    print(f"\n{'='*60}")
    print(f"🚀 BẮT ĐẦU NẠP & LỌC NGÀNH HÀNG: {category}")
    print(f"{'='*60}")

    # Tạo thư mục nếu nó chưa tồn tại trên máy
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Khởi tạo đường dẫn gốc từ Hugging Face (Dùng HTTPS để tránh lỗi URL)
    review_https_url = f"https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/{category}.jsonl"
    meta_https_url = f"https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/meta_categories/meta_{category}.jsonl"

    # Nối tên file vào đường dẫn thư mục của bạn
    review_parquet = os.path.join(OUTPUT_DIR, f"raw_review_{category}.parquet")
    meta_parquet = os.path.join(OUTPUT_DIR, f"raw_meta_{category}.parquet")

    # ------------------------------------------
    # A. XỬ LÝ DỮ LIỆU REVIEW
    # ------------------------------------------
    if not os.path.exists(review_parquet):
        print(f"⏳ Đang tải và lọc Review data vào:\n   -> {review_parquet}")
        try:
            (
                pl.scan_ndjson(review_https_url, infer_schema_length=10000)
                # CHỈ LỌC CÁC CỘT CẦN DÙNG
                .select([
                    "parent_asin", 
                    "user_id", 
                    "rating", 
                    "text", 
                    "timestamp", 
                    "verified_purchase", 
                    "helpful_vote"
                ])
                .sink_parquet(review_parquet) # Chuyển đổi và lưu
            )
            print(f"✅ Đã lưu xong Review: {category}")
        except Exception as e:
            print(f"❌ Lỗi khi tải Review {category}: {e}")
    else:
        print(f"⏭️ Bỏ qua Review: File đã tồn tại ở {review_parquet}")

    # ------------------------------------------
    # B. XỬ LÝ DỮ LIỆU METADATA
    # ------------------------------------------
    if not os.path.exists(meta_parquet):
        print(f"⏳ Đang tải và lọc Meta data vào:\n   -> {meta_parquet}")
        try:
            (
                pl.scan_ndjson(meta_https_url, infer_schema_length=10000)
                # CHỈ LỌC CÁC CỘT CẦN DÙNG
                .select([
                    "parent_asin", 
                    "main_category", 
                    "title"
                ])
                .sink_parquet(meta_parquet) # Chuyển đổi và lưu
            )
            print(f"✅ Đã lưu xong Meta: {category}")
        except Exception as e:
            print(f"❌ Lỗi khi tải Meta {category}: {e}")
    else:
        print(f"⏭️ Bỏ qua Meta: File đã tồn tại ở {meta_parquet}")

In [ ]:
# ==========================================
# THỰC THI
# ==========================================
if __name__ == "__main__":
    for cat in CATEGORIES:
        ingest_https_to_parquet(cat)
    
    print(f"\n🎉 HOÀN TẤT! Hãy mở ổ D của bạn để kiểm tra dữ liệu.")

In [ ]:
# file Clothing_Shoes_and_Jewelry
import polars as pl
from datasets import load_dataset
import pyarrow.parquet as pq
import os

# ==========================================
# CẤU HÌNH
# ==========================================
CATEGORY = "Clothing_Shoes_and_Jewelry"
OUTPUT_DIR = r"D:\Kì II 2025-2026\BTL_BIGDATA\Data"
BATCH_SIZE = 100000  # Xử lý 100.000 dòng mỗi lần để không tràn RAM

# Các cột cần giữ lại
REVIEW_COLS = ["parent_asin", "user_id", "rating", "text", "timestamp", "verified_purchase", "helpful_vote"]
META_COLS = ["parent_asin", "main_category", "title"]

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# HÀM XỬ LÝ STREAMING TỪNG LÔ (BATCHING)
# ==========================================
def stream_and_compress(subset_name, output_file, columns_to_keep):
    print(f"\n⏳ Bắt đầu stream và nén: {subset_name}")
    print(f"   -> Đích đến: {output_file}")
    
    if os.path.exists(output_file):
        print(f"⏭️ File đã tồn tại, bỏ qua!")
        return

    dataset = load_dataset(
        "McAuley-Lab/Amazon-Reviews-2023", 
        subset_name, 
        split="full", 
        streaming=True, 
        trust_remote_code=True
    )
    
    writer = None
    batch = []
    total_rows = 0

    try:
        for row in dataset:
            # LỌC CỘT
            filtered_row = {col: row.get(col) for col in columns_to_keep}
            batch.append(filtered_row)
            
            # Khi đủ 1 lô (100.000 dòng), tiến hành nén và ghi ra ổ cứng
            if len(batch) >= BATCH_SIZE:
                df = pl.from_dicts(batch)
                table = df.to_arrow()
                
                # Nếu là lô đầu tiên, tạo cấu trúc file Parquet
                if writer is None:
                    writer = pq.ParquetWriter(output_file, table.schema)
                
                # Ghi nối tiếp vào file
                writer.write_table(table)
                total_rows += len(batch)
                print(f"   ... Đã ghi {total_rows:,} dòng ...")
                
                # Xóa lô hiện tại khỏi RAM để nhường chỗ cho lô mới
                batch = []

        # Ghi nốt phần dữ liệu lẻ còn sót lại ở cuối file
        if len(batch) > 0:
            df = pl.from_dicts(batch)
            table = df.to_arrow()
            if writer is None:
                writer = pq.ParquetWriter(output_file, table.schema)
            writer.write_table(table)
            total_rows += len(batch)
            print(f"   ... Đã ghi {total_rows:,} dòng ...")

    finally:
        # Đóng file an toàn
        if writer:
            writer.close()
            
    print(f"✅ HOÀN TẤT! Đã lưu tổng cộng {total_rows:,} dòng siêu nhẹ vào ổ đĩa.\n")

# ==========================================
# THỰC THI (Chỉ dành cho Clothing)
# ==========================================
if __name__ == "__main__":
    print("🚀 TIẾU DIỆT TRÙM CUỐI CLOTHING & SHOES...")
    
    # 1. Tải Review
    review_subset = f"raw_review_{CATEGORY}"
    review_output = os.path.join(OUTPUT_DIR, f"raw_review_{CATEGORY}.parquet")
    stream_and_compress(review_subset, review_output, REVIEW_COLS)

    # 2. Tải Meta
    meta_subset = f"raw_meta_{CATEGORY}"
    meta_output = os.path.join(OUTPUT_DIR, f"raw_meta_{CATEGORY}.parquet")
    stream_and_compress(meta_subset, meta_output, META_COLS)
    
    print("🎉 CHÚC MỪNG! BẠN ĐÃ THU THẬP XONG TOÀN BỘ DỮ LIỆU NHÓM 3!")

d:\dowloadd\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 TIẾU DIỆT TRÙM CUỐI CLOTHING & SHOES...

⏳ Bắt đầu stream và nén: raw_review_Clothing_Shoes_and_Jewelry
   -> Đích đến: D:\Kì II 2025-2026\BTL_BIGDATA\Data\raw_review_Clothing_Shoes_and_Jewelry.parquet
   ... Đã ghi 100,000 dòng ...
   ... Đã ghi 200,000 dòng ...
   ... Đã ghi 300,000 dòng ...
   ... Đã ghi 400,000 dòng ...
   ... Đã ghi 500,000 dòng ...
   ... Đã ghi 600,000 dòng ...
   ... Đã ghi 700,000 dòng ...
   ... Đã ghi 800,000 dòng ...
   ... Đã ghi 900,000 dòng ...
   ... Đã ghi 1,000,000 dòng ...
   ... Đã ghi 1,100,000 dòng ...
   ... Đã ghi 1,200,000 dòng ...
   ... Đã ghi 1,300,000 dòng ...
   ... Đã ghi 1,400,000 dòng ...
   ... Đã ghi 1,500,000 dòng ...
   ... Đã ghi 1,600,000 dòng ...
   ... Đã ghi 1,700,000 dòng ...
   ... Đã ghi 1,800,000 dòng ...
   ... Đã ghi 1,900,000 dòng ...
   ... Đã ghi 2,000,000 dòng ...
   ... Đã ghi 2,100,000 dòng ...
   ... Đã ghi 2,200,000 dòng ...
   ... Đã ghi 2,300,000 dòng ...
   ... Đã ghi 2,400,000 dòng ...
   ... Đã ghi 2,500,00

'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/Clothing_Shoes_and_Jewelry.jsonl
Retrying in 1s [Retry 1/5].
'timed out' thrown while requesting GET https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/Clothing_Shoes_and_Jewelry.jsonl
Retrying in 2s [Retry 2/5].


   ... Đã ghi 3,300,000 dòng ...
   ... Đã ghi 3,400,000 dòng ...
   ... Đã ghi 3,500,000 dòng ...
   ... Đã ghi 3,600,000 dòng ...
   ... Đã ghi 3,700,000 dòng ...
   ... Đã ghi 3,800,000 dòng ...
   ... Đã ghi 3,900,000 dòng ...
   ... Đã ghi 4,000,000 dòng ...
   ... Đã ghi 4,100,000 dòng ...
   ... Đã ghi 4,200,000 dòng ...
   ... Đã ghi 4,300,000 dòng ...
   ... Đã ghi 4,400,000 dòng ...
   ... Đã ghi 4,500,000 dòng ...
   ... Đã ghi 4,600,000 dòng ...
   ... Đã ghi 4,700,000 dòng ...
   ... Đã ghi 4,800,000 dòng ...
   ... Đã ghi 4,900,000 dòng ...
   ... Đã ghi 5,000,000 dòng ...
   ... Đã ghi 5,100,000 dòng ...
   ... Đã ghi 5,200,000 dòng ...
   ... Đã ghi 5,300,000 dòng ...
   ... Đã ghi 5,400,000 dòng ...
   ... Đã ghi 5,500,000 dòng ...
   ... Đã ghi 5,600,000 dòng ...
   ... Đã ghi 5,700,000 dòng ...
   ... Đã ghi 5,800,000 dòng ...
   ... Đã ghi 5,900,000 dòng ...
   ... Đã ghi 6,000,000 dòng ...
   ... Đã ghi 6,100,000 dòng ...
   ... Đã ghi 6,200,000 dòng ...
   ... Đã 